In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [2]:
from sklearn.preprocessing import OneHotEncoder
training_file_path='/kaggle/input/competitions/home-data-for-ml-course/train.csv'
training_data = pd.read_csv(training_file_path)

t = training_data.SalePrice
X = training_data.drop(['SalePrice'],axis = 1)
missing_cols = X.isna().sum()
print(missing_cols[missing_cols > 0])
redflags = ['Alley','MasVnrType','FireplaceQu','PoolQC','Fence','MiscFeature']
X = X.drop(redflags,axis=1)
X = X.drop('MiscVal',axis=1)
print(X.columns)
num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns
for col in num_cols:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)

for col in cat_cols:
    mode_val = X[col].mode()[0]
    X[col] = X[col].fillna(mode_val)
onehotlist = ['MSSubClass','MSZoning','Street', 'LandContour', 'LotConfig', 'Neighborhood',
              'Condition1','Condition2','BldgType', 'HouseStyle', 'RoofStyle','RoofMatl',
              'Exterior1st','Exterior2nd','Foundation','Heating','CentralAir','GarageType','SaleType','SaleCondition']
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = False)
OH_cols = pd.DataFrame(OH_encoder.fit_transform(X[onehotlist]))
OH_cols.index = X.index
num_X = X.drop(onehotlist,axis=1)
OH_X = pd.concat([num_X,OH_cols],axis=1)
OH_X.columns = OH_X.columns.astype(str)
#OH_X.head

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64
Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'Central

In [3]:
X.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF',
       'Enclo

In [4]:
from sklearn.preprocessing import OrdinalEncoder

reference = ['MSSubClass','MSZoning','Street', 'LandContour', 'LotConfig', 'Neighborhood',
              'Condition1','Condition2','BldgType', 'HouseStyle', 'RoofStyle','RoofMatl',
              'Exterior1st','Exterior2nd','Foundation','Heating','CentralAir','GarageType','SaleType','SaleCondition']

ordinalCols = ['LotShape', 'Utilities','LandSlope', 'ExterQual','ExterCond','BsmtQual',
               'BsmtCond', 'BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC',
               'Electrical','KitchenQual','Functional','GarageFinish', 'GarageQual',
               'GarageCond','PavedDrive']
lot_ranks = ["IR3","IR2","IR1","Reg"]
slope_ranks = ["Sev","Mod","Gtl"]
utility_ranks = ['ELO','NoSeWa','NoSewr','Normal','AllPub']
exposure_ranks = ['No','Mn','Av','Gd']
quality_ranks = ["Po","Fa","TA","Gd","Ex"]
paved_ranks = ["N","P","Y"]
finish_ranks = ['Unf','LwQ','Rec','BLQ','ALQ','GLQ']
electrical_ranks = ["FuseP", "FuseF", "FuseA", "Mix", "SBrkr"]
functional_ranks = ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ']
gfinish_ranks = ['NA','Unf','RFn','Fin']

categories_list= [
    lot_ranks,
    utility_ranks,
    slope_ranks,
    quality_ranks,
    quality_ranks,
    quality_ranks,
    quality_ranks,
    exposure_ranks,
    finish_ranks,
    finish_ranks,
    quality_ranks,
    electrical_ranks,
    quality_ranks,
    functional_ranks,
    gfinish_ranks,
    quality_ranks,
    quality_ranks,
    paved_ranks
]
ordinal_encoder = OrdinalEncoder(categories = categories_list)



In [5]:
final_X = OH_X.copy()
final_X[ordinalCols]= ordinal_encoder.fit_transform(OH_X[ordinalCols])
final_X = final_X.drop(["Id"],axis=1)
train_mean = np.mean(final_X, axis=0)
train_std = np.std(final_X, axis=0)
final_X = (final_X - train_mean) / train_std
final_X.shape

(1460, 218)

In [6]:
weights = np.zeros(218)
bias = 0
learning_rate= 0.0085
reg_parameter = 300
for i in range(2200):
    y = final_X@weights+bias
  
 
    weights = weights-((learning_rate)/1460)*((final_X.T)@(y-t))-(1/1460)*learning_rate*reg_parameter*weights
    bias =  bias-(1/1460)*learning_rate*((y-t).sum())
print(bias)
print(weights)


    

180921.19462721646
LotFrontage    1734.638243
LotArea        3271.972828
LotShape       -345.747484
Utilities       975.774766
LandSlope      -483.413953
                  ...     
161             317.824327
162             247.378068
163           -1009.272113
164            -111.568526
165            1573.233676
Length: 218, dtype: float64


In [7]:
import pandas as pd
import numpy as np

# 1. Load test data
test_file_path = '/kaggle/input/competitions/home-data-for-ml-course/test.csv'
test_data = pd.read_csv(test_file_path)

# Save 'Id' column for the submission file
test_ids = test_data['Id']

# Keep only columns present in your training set (X)
test_X = test_data[X.columns].copy()
for col in num_cols:
    median_val = X[col].median()
    test_X[col] = test_X[col].fillna(median_val)

for col in cat_cols:
    mode_val = X[col].mode()[0]
    test_X[col] = test_X[col].fillna(mode_val)

OH_cols_test = pd.DataFrame(OH_encoder.transform(test_X[onehotlist]))
OH_cols_test.index = test_X.index

num_test_X = test_X.drop(onehotlist, axis=1)
OH_test_X = pd.concat([num_test_X, OH_cols_test], axis=1)
OH_test_X.columns = OH_test_X.columns.astype(str)

# 3. Apply OrdinalEncoder (.transform ONLY)
final_test_X = OH_test_X.copy()
final_test_X[ordinalCols] = ordinal_encoder.transform(OH_test_X[ordinalCols])
final_test_X = final_test_X.drop(["Id"], axis=1, errors='ignore')

final_test_X = (final_test_X - train_mean) / train_std

test_preds = final_test_X @ weights 
test_preds +=bias
print(test_preds)

output = pd.DataFrame({'Id': test_ids, 'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)
print("Submission saved successfully!")

0       113968.359699
1       169518.573593
2       179351.154787
3       190293.024207
4       216542.152091
            ...      
1454     64137.531783
1455     69944.004992
1456    170392.385204
1457    102359.426610
1458    223363.651828
Length: 1459, dtype: float64
Submission saved successfully!
